# Preprocessing CBIS-DDSM dataset

# Important libraries

# Preprocessing class with all needed functions

-------------------------------------------------------------------------------------------------------------------------------------------------------

-------------------------------------------------------------------------------------------------------------------------------------------------------

In [2]:
pip install pydicom

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.4 MB 1.4 MB/s eta 0:00:02
   ------------- -------------------------- 0.8/2.4 MB 1.5 MB/s eta 0:00:02
   ---------------------- ----------------- 1.3/2.4 MB 1.7 MB/s eta 0:00:01
   -------------------------- ------------- 1.6/2.4 MB 1.7 MB/s eta 0:00:01
   ------------------------------ --------- 1.8/2.4 MB 1.6 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 1.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os
import pandas as pd
import pydicom
import numpy as np
import cv2  # OpenCV for image resizing and saving

# === CONFIG ===
RAW_DATA_ROOT = "C:/Users/DragosTrandafiri/BreastCancer_CNN_bachelor_thesis/data/raw/manifest-1759485901847"
METADATA_CSV = "C:/Users/DragosTrandafiri/BreastCancer_CNN_bachelor_thesis/data/raw/manifest-1759485901847/metadata.csv"
OUTPUT_DIR = "C:/Users/DragosTrandafiri/BreastCancer_CNN_bachelor_thesis/data/processed"
IMAGE_SIZE = (224, 224)

# Create output directory if not exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === LOAD METADATA ===
df = pd.read_csv(METADATA_CSV)

# Filter only full mammogram images
full_mammo_df = df[df["Series Description"] == "full mammogram images"]

# === PROCESS IMAGES ===
for idx, row in full_mammo_df.iterrows():
    subject_id = row["Subject ID"]
    rel_path = row["File Location"]
    
    # Normalize path
    rel_path = rel_path.replace(".\\", "").replace("/", "\\")  # fix relative path format

    dicom_dir = os.path.join(RAW_DATA_ROOT, rel_path)

    if not os.path.exists(dicom_dir):
        print(f"[WARNING] Directory not found: {dicom_dir}")
        continue

    for file in os.listdir(dicom_dir):
        if file.endswith(".dcm"):
            dicom_path = os.path.join(dicom_dir, file)
            try:
                # Load DICOM
                ds = pydicom.dcmread(dicom_path)
                pixel_array = ds.pixel_array

                # Normalize pixel values to 0-255
                pixel_array = pixel_array.astype(np.float32)
                pixel_array -= pixel_array.min()
                pixel_array /= pixel_array.max()
                pixel_array *= 255.0
                pixel_array = pixel_array.astype(np.uint8)

                # Resize to 224x224
                resized = cv2.resize(pixel_array, IMAGE_SIZE, interpolation=cv2.INTER_AREA)

                # Save as PNG
                save_filename = f"{subject_id}_{file.replace('.dcm', '')}.png"
                save_path = os.path.join(OUTPUT_DIR, save_filename)
                cv2.imwrite(save_path, resized)

                print(f"[INFO] Saved: {save_path}")

            except Exception as e:
                print(f"[ERROR] Could not process {dicom_path}: {e}")


[INFO] Saved: C:/Users/DragosTrandafiri/BreastCancer_CNN_bachelor_thesis/data/processed\Calc-Test_P_00038_LEFT_MLO_1-1.png
[INFO] Saved: C:/Users/DragosTrandafiri/BreastCancer_CNN_bachelor_thesis/data/processed\Calc-Test_P_00038_LEFT_CC_1-1.png
[INFO] Saved: C:/Users/DragosTrandafiri/BreastCancer_CNN_bachelor_thesis/data/processed\Calc-Test_P_00038_RIGHT_CC_1-1.png
[INFO] Saved: C:/Users/DragosTrandafiri/BreastCancer_CNN_bachelor_thesis/data/processed\Calc-Test_P_00038_RIGHT_MLO_1-1.png
[INFO] Saved: C:/Users/DragosTrandafiri/BreastCancer_CNN_bachelor_thesis/data/processed\Calc-Test_P_00041_LEFT_CC_1-1.png
[INFO] Saved: C:/Users/DragosTrandafiri/BreastCancer_CNN_bachelor_thesis/data/processed\Calc-Test_P_00041_LEFT_MLO_1-1.png
[INFO] Saved: C:/Users/DragosTrandafiri/BreastCancer_CNN_bachelor_thesis/data/processed\Calc-Test_P_00077_LEFT_CC_1-1.png
[INFO] Saved: C:/Users/DragosTrandafiri/BreastCancer_CNN_bachelor_thesis/data/processed\Calc-Test_P_00077_RIGHT_CC_1-1.png
[INFO] Saved: C:/U